# पाठ 05 - एजेंटिक RAG


## सेटअप

यह नोटबुक Microsoft Agent Framework का उपयोग करके Agentic RAG (Retrieval-Augmented Generation) पैटर्न प्रदर्शित करता है।

**पूर्वापेक्षाएँ:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — आपका Azure AI Search सेवा एंडपॉइंट
- `AZURE_SEARCH_API_KEY` — आपका Azure AI Search API कुंजी
- पर्यावरण चर के माध्यम से कॉन्फ़िगर किया गया Azure OpenAI डिप्लॉयमेंट
- Azure CLI प्रमाणीकृत (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Agentic RAG क्या है?

पारंपरिक RAG एक निश्चित प्रक्रिया का पालन करता है: दस्तावेज़ प्राप्त करें, फिर एक उत्तर उत्पन्न करें। **Agentic RAG** इससे आगे बढ़ता है और एजेंट को यह स्वयं निर्णय लेने की स्वतंत्रता देता है कि **कब** और **कैसे** जानकारी प्राप्त करनी है।

Agentic RAG के साथ, एजेंट कर सकता है:
- **निर्णय लें** कि किसी प्रश्न का उत्तर देने से पहले जानकारी प्राप्त करना आवश्यक है या नहीं
- **चुनें** कि किस डेटा स्रोत या उपकरण से पूछताछ करनी है
- **मूल्यांकन करें** प्राप्त परिणामों का और यदि पहली कोशिश अपर्याप्त हो तो पुनः जानकारी प्राप्त करें
- **मिलाएँ** विभिन्न प्राप्त चरणों से जानकारी को एक सुसंगत उत्तर में

यह एजेंट को एक स्थिर प्राप्त-फिर-उत्पन्न प्रक्रिया की तुलना में अधिक लचीला और सटीक बनाता है।


## एक खोज उपकरण बनाना

Agentic RAG में, बाहरी डेटा स्रोतों को **उपकरणों** के रूप में लपेटा जाता है जिन्हें एजेंट आवश्यकता अनुसार कॉल कर सकता है। इससे एजेंट पुनः प्राप्ति को केवल एक और क्रिया के रूप में देख सकता है, बजाय इसके कि वह एक अनिवार्य कदम हो।

नीचे हम एक यात्रा ज्ञान आधार को परिभाषित करते हैं और इसे एक उपकरण के रूप में प्रस्तुत करते हैं जिसे एजेंट गंतव्य जानकारी खोजने के लिए कॉल कर सकता है।


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG एजेंट का निर्माण

अब हम एक एजेंट तैयार करते हैं जिसे निर्देश दिया गया है कि **हमेशा उत्तर देने से पहले जानकारी प्राप्त करे**। एजेंट अपने जवाबों को अपने प्रशिक्षण डेटा पर भरोसा करने के बजाय ज्ञान बेस में आधारित करने के लिए `search_travel_knowledge` टूल का उपयोग करता है।


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## पुनरावृत्त पुनः प्राप्ति — मेकर-चेकर पैटर्न

Agentic RAG का एक प्रमुख लाभ है **पुनरावृत्त पुनः प्राप्ति**। एजेंट कई बार खोज कर सकता है ताकि अपनी प्रारंभिक खोजों को सत्यापित, परिष्कृत या विस्तारित कर सके — जो कि "मेकर-चेकर" कार्यप्रवाह के समान है:

1. **मेकर चरण**: एजेंट प्रारंभिक जानकारी पुनः प्राप्त करता है और एक उत्तर का मसौदा तैयार करता है।
2. **चेकर चरण**: एजेंट विवरणों को सत्यापित करने या खाली स्थान भरने के लिए अतिरिक्त पुनः प्राप्ति करता है।

नीचे, एजेंट से ऐसा प्रश्न पूछा गया है जिसमें कई गंतव्यों की तुलना करनी होती है, जिससे वह कई बार खोज करने के लिए प्रेरित होता है।


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## सारांश

इस पाठ में आपने Microsoft Agent Framework का उपयोग करके एक **Agentic RAG** सिस्टम बनाना सीखा:

- **Agentic RAG** एजेंट्स को स्वायत्त रूप से यह निर्णय लेने देता है कि जानकारी कब प्राप्त करनी है, जिससे पुनःप्राप्ति निश्चित के बजाय गतिशील हो जाती है।
- **डेटा स्रोतों के रूप में टूल्स**: बाहरी ज्ञान आधार (जैसे Azure AI Search) को ऐसे टूल्स के रूप में लपेटा गया है जिन्हें एजेंट कॉल कर सकता है।
- **पुनरावृत्त पुनःप्राप्ति**: मेकर-चेकर पैटर्न एजेंट को कई पुनःप्राप्ति राउंड करने में सक्षम बनाता है — खोजने, सत्यापित करने, और परिष्कृत करने — अंतिम उत्तर देने से पहले।

उत्पादन में, आप in-memory `TRAVEL_KNOWLEDGE_BASE` को बड़े स्तर पर यात्रा दस्तावेज़ पुनःप्राप्ति को संभालने के लिए एक वास्तविक Azure AI Search सूचकांक से बदल देंगे।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:  
यह दस्तावेज़ एआई अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके अनिवारित किया गया है। जबकि हम सटीकता के लिए प्रयासरत हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियां या गलतियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही आधिकारिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सलाह दी जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या भ्रामक व्याख्या के लिए हम जिम्मेदार नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
